In [5]:
import json
import pandas as pd
from sklearn.model_selection import train_test_split

# Load dataset
DATA_PATH = r"../datasets/food_dataset_extended.json"
with open(DATA_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)

# Normalize occasion field
df["occasion_text"] = df["occasion"].apply(
    lambda x: ", ".join(x) if isinstance(x, list) else str(x)
)
df["type"] = df["type"].fillna("Dish")

# Filter by dietary preference if needed
df_veg = df[df["dietary_preference"] == "Vegetarian"].copy()

# Build structured menus with flexible requirements
def build_structured_menu(subset, target_starters=3, target_mains=4, target_desserts=2):
    """Generate menu with flexible dish counts based on available data"""
    
    # Categorize dishes
    starters = subset[
        subset["type"].str.contains("Starter|Appetizer|Snack", case=False, na=False)
    ]["item_name"].unique().tolist()
    
    beverages = subset[
        subset["type"].str.contains("Beverage", case=False, na=False)
    ]["item_name"].unique().tolist()
    
    mains = subset[
        subset["type"].str.contains("Main|Curry|Rice|Biryani|Bread|Dish", case=False, na=False)
    ]["item_name"].unique().tolist()
    
    desserts = subset[
        subset["type"].str.contains("Dessert|Sweet", case=False, na=False)
    ]["item_name"].unique().tolist()
    
    # Combine starters and beverages
    starters_combined = list(set(starters + beverages))
    
    # Get all unique dishes as fallback
    all_dishes = subset["item_name"].unique().tolist()
    
    # Flexible padding - use what's available
    def get_dishes(preferred, fallback, target_count):
        """Get dishes, using fallback if needed"""
        result = preferred[:target_count]
        
        # If not enough, add from fallback (avoiding duplicates)
        if len(result) < target_count:
            used = set(result)
            for dish in fallback:
                if dish not in used and len(result) < target_count:
                    result.append(dish)
                    used.add(dish)
        
        return result
    
    # Build sections with fallback to all dishes
    starters_final = get_dishes(starters_combined, all_dishes, target_starters)
    mains_final = get_dishes(mains, all_dishes, target_mains)
    desserts_final = get_dishes(desserts, all_dishes, target_desserts)
    
    # Only create menu if we have minimum viable data
    if len(starters_final) < 1 or len(mains_final) < 2 or len(desserts_final) < 1:
        return None
    
    # Format menu
    menu = (
        "Starters\n" + "\n".join(f"- {d}" for d in starters_final) + "\n\n" +
        "Main Course\n" + "\n".join(f"- {d}" for d in mains_final) + "\n\n" +
        "Dessert\n" + "\n".join(f"- {d}" for d in desserts_final)
    )
    
    return menu.strip()

# Generate training examples with better error handling
# Group by just one field: occasion_text
menu_records = []
skipped_groups = 0

for occasion_text, group in df.groupby('occasion_text'):
    if len(group) < 3:
        skipped_groups += 1
        continue

    menu_text = build_structured_menu(group, target_starters=3, target_mains=4, target_desserts=2)
    if menu_text is None:
        skipped_groups += 1
        continue

    actual_starters = menu_text.count("Starters\n") and len([l for l in menu_text.split("Main Course")[0].split("\n") if l.strip().startswith("-")])
    actual_mains = len([l for l in menu_text.split("Main Course\n")[1].split("Dessert")[0].split("\n") if l.strip().startswith("-")])
    actual_desserts = len([l for l in menu_text.split("Dessert\n")[1].split("\n") if l.strip().startswith("-")])

    input_text = (
        f"Generate a vegetarian menu for {occasion_text}. "
        f"Include: {actual_starters} starters, {actual_mains} main course dishes, {actual_desserts} desserts."
    )

    menu_records.append({
        "input_text": input_text,
        "target_text": menu_text,
        "occasion": occasion_text,
    })

print(f"Skipped groups: {skipped_groups}")
print(f"Valid training examples created: {len(menu_records)}")

menu_df = pd.DataFrame(menu_records)
if len(menu_df) == 0:
    print("❌ Still not enough items per occasion! Try the next fallback.")
# Split dataset
train_df, eval_df = train_test_split(menu_df, test_size=0.1, random_state=42)


Skipped groups: 198
Valid training examples created: 6


In [6]:
from transformers import T5Tokenizer, T5ForConditionalGeneration
from torch.utils.data import Dataset
import torch

tokenizer = T5Tokenizer.from_pretrained("t5-small")
model = T5ForConditionalGeneration.from_pretrained("t5-small")

class MenuDataset(Dataset):
    def __init__(self, df, tokenizer, max_input_length=256, max_target_length=350):
        self.tokenizer = tokenizer
        self.inputs = list(df["input_text"])
        self.targets = list(df["target_text"])
        self.max_input_length = max_input_length
        self.max_target_length = max_target_length
    
    def __len__(self):
        return len(self.inputs)
    
    def __getitem__(self, idx):
        input_text = self.inputs[idx]
        target_text = self.targets[idx]
        
        # Tokenize input
        input_encoding = self.tokenizer(
            input_text,
            max_length=self.max_input_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        
        # Tokenize target
        target_encoding = self.tokenizer(
            target_text,
            max_length=self.max_target_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        
        labels = target_encoding["input_ids"].squeeze()
        # Replace padding token id with -100 (ignored in loss)
        labels[labels == self.tokenizer.pad_token_id] = -100
        
        return {
            "input_ids": input_encoding["input_ids"].squeeze(),
            "attention_mask": input_encoding["attention_mask"].squeeze(),
            "labels": labels
        }

train_dataset = MenuDataset(train_df, tokenizer)
eval_dataset = MenuDataset(eval_df, tokenizer)


d:\Projects\SPC\spc\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./menu_generator_model",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=10,
    learning_rate=3e-4,
    weight_decay=0.01,
    save_total_limit=2,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    warmup_steps=100,
    fp16=True,  # Enable if you have GPU
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)


In [9]:
# Train
trainer.train()

d:\Projects\SPC\spc\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss
1,No log,3.118396
2,No log,3.097989
3,No log,3.061241
4,No log,3.015171
5,No log,2.959942
6,No log,2.902426
7,No log,2.854791
8,No log,2.803379
9,No log,2.746086
10,No log,2.698042


d:\Projects\SPC\spc\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
d:\Projects\SPC\spc\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
d:\Projects\SPC\spc\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
d:\Projects\SPC\spc\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
d:\Projects\SPC\spc\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as t

TrainOutput(global_step=20, training_loss=3.5391178131103516, metrics={'train_runtime': 49.9927, 'train_samples_per_second': 1.0, 'train_steps_per_second': 0.4, 'total_flos': 3383545036800.0, 'train_loss': 3.5391178131103516, 'epoch': 10.0})

In [ ]:
# Save
model.save_pretrained("./trained_menu_model")
tokenizer.save_pretrained("./trained_menu_model")

In [ ]:
def generate_menu(occasion, cuisine, model, tokenizer, vegetarian=True):
    veg_text = "vegetarian " if vegetarian else ""
    
    # Match training input format EXACTLY
    prompt = (
        f"Generate a {veg_text}menu for {occasion} in {cuisine} cuisine. "
        f"Include: 3 starters, 4 main course dishes, 2 desserts."
    )
    
    inputs = tokenizer(prompt, return_tensors="pt", max_length=256, truncation=True)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=350,
            min_new_tokens=120,
            num_beams=5,
            early_stopping=True,
            no_repeat_ngram_size=3,
            repetition_penalty=1.2,
            length_penalty=1.0,
            pad_token_id=tokenizer.eos_token_id
        )
    
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return result

# Test
print(generate_menu("Corporate Lunch", "Indian (North)", model, tokenizer))


Starters Main Course - Vegetable Pesarattu Main Course Dessert - Pesatu-Pièce du Pontifical Meal - Veg Pesée Jasal Pesar - Dip Dip Doubtu Desserts - Goeda Kadhi Pakora Paneer Tadjia-Kidjong Tadka Desserts - Do not eat a lot of meat, but - pesa rattis) Minuteing - I Plat
